In [40]:
# import vari
import numpy as np
import pandas as pd
import matplotlib.pyplot as pt
import seaborn as sb

In [41]:
samplesize = 0.05
sample = pd.read_pickle(f"./data/sample_{samplesize}.pkl")
dataset = sample

In [42]:
# following e follower count hanno valori letterali, vediamo quali sono
fgc = dataset["following_count"]
fgcNumeric = pd.to_numeric(fgc, errors="coerce")

fgcInvalid = fgc[fgcNumeric.isna() & fgc.notna()]
fgcInvalid.value_counts()



following_count
error      17
many       13
unknown    11
none       11
ten        11
NaN-ish    11
??42       10
12abc       8
4O00        7
1,2,3       2
Name: count, dtype: int64

In [43]:
frc = dataset["follower_count"]
frcNumeric = pd.to_numeric(frc, errors="coerce")

frcInvalid = frc[frcNumeric.isna() & frc.notna()]
frcInvalid.value_counts()


follower_count
error      17
none       17
many       16
NaN-ish    14
unknown    14
??42       13
4O00       12
12abc      11
ten         8
1,2,3       6
Name: count, dtype: int64

In [44]:
# sono relativamente pochi. Li potremmo normalizzare, magari sostituendo 0 a none e un qualche valore calcolato
# dagli altri record su many o unknown, ma è meglio non fare operazioni distruttive, e aggiungere una colonna
# che porti i valori ripuliti

In [45]:
dataset["following_count_clean"] = fgcNumeric.astype("Int64")
dataset["follower_count_clean"] = frcNumeric.astype("Int64")

In [46]:
# ci sono dei booleani scritti come stringhe, e un  is_control?, che non si sa se mettere true o false
# anche in questo caso, in una nuova colonna
dataset["is_control"].map(lambda v: (str(v), type(v).__name__)).value_counts()

dataset["is_control_clean"] = dataset["is_control"].map(
    {True: True, "True": True, False: False, "False": False}
).astype("boolean")

In [47]:
dataset["is_control_clean"].unique()

<BooleanArray>
[True, False, <NA>]
Length: 3, dtype: boolean

In [55]:
# trasforma da stringa a datetime
dataset["post_time_clean"] = pd.to_datetime(dataset["post_time"], format="mixed", errors="coerce")
# dataset["post_time_clean"].isna().any()

dataset["account_creation_date_clean"] = pd.to_datetime(dataset["account_creation_date"], format="mixed", errors="coerce")
# dataset["account_creation_date_clean"].isna().any()

In [ ]:
# nel post_text le menzioni sono discutibilmente testo semanticamente significativo, ma gonfiano molto la lunghezza
# quindi ha senso eliminarle (e analizzare in seguito) 
dataset["post_text_clean"]= dataset["post_text"].str.replace(r"@\w+", "", regex=True).str.len()

In [56]:
dataset.to_pickle(f"./data/sample_{samplesize}_clean.pkl")